# Hypothesis Space II: From Fixed Features to Deep Representations


## 0. Learning objectives and workshop frame

This notebook isolates $\mathcal{H}$: the set of functions made available by modelling choices. We keep data and optimisation in view, but the main question is what assumptions enter when we choose a feature map, activation, width, depth, or architecture.

The workshop frame remains

$$
\mathcal{H}+\mathcal{D}+\mathcal{O}\rightarrow s.
$$

Previously, a learner selected a function from a fixed hypothesis space

$$
\mathcal{H}_{\phi}
= \left\{x \mapsto \theta^{\top}\phi(x) : \theta \in \Theta\right\}.
$$

In deep learning, the representation is also parameterised:

$$
\mathcal{H}_{\mathrm{NN}}
= \left\{x \mapsto h_{\theta}(x) : \theta = (W_1,b_1,\ldots,W_L,b_L)\right\}.
$$

We will ask four questions repeatedly:

1. What functions are possible?
2. What functions are easy or natural for this architecture?
3. Where do functions agree because of data, and where do they agree only because of bias?
4. What extrapolation behaviour is being assumed?

By the end, students should be able to distinguish expressivity, approximation, optimisation, identification, and generalisation as separate parts of a research claim.


In [ ]:
from pathlib import Path
import sys

from IPython.display import Markdown, display

PROJECT_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src" / "nextgen2026_mlai_workshops").exists()),
    Path.cwd(),
)
src_dir = PROJECT_ROOT / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

try:
    import ipywidgets as widgets
    HAS_WIDGETS = True
except Exception:
    widgets = None
    HAS_WIDGETS = False

from nextgen2026_mlai_workshops.data_space import configure_matplotlib
from nextgen2026_mlai_workshops.hypothesis_space import (
    fixed_vs_learned_action_rows,
    plot_architecture_demo,
    plot_capacity_demo,
    plot_depth_demo,
    plot_fixed_vs_learned,
    plot_parameter_equivalence,
    plot_relu_neuron,
    plot_support_model,
    plot_width_demo,
    summary_rows,
    what_changed_rows,
)

configure_matplotlib()


def markdown_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |", "|" + "|".join(["---"] * len(headers)) + "|"]
    for row in rows:
        lines.append("| " + " | ".join(str(item) for item in row) + " |")
    display(Markdown("\n".join(lines)))


def slider(kind, **kwargs):
    if not HAS_WIDGETS:
        return None
    kwargs.setdefault("style", {"description_width": "initial"})
    return getattr(widgets, kind)(**kwargs)


def dropdown(**kwargs):
    if not HAS_WIDGETS:
        return None
    kwargs.setdefault("style", {"description_width": "initial"})
    return widgets.Dropdown(**kwargs)


def checkbox(**kwargs):
    if not HAS_WIDGETS:
        return None
    kwargs.setdefault("style", {"description_width": "initial"})
    return widgets.Checkbox(**kwargs)


def show_interactive(guidance, render_fn, controls, fallback):
    display(Markdown(f"> **Investigate.** {guidance}"))
    if HAS_WIDGETS:
        ui = widgets.VBox(list(controls.values()))
        out = widgets.interactive_output(render_fn, controls)
        display(ui, out)
    else:
        display(Markdown("> Widgets are unavailable in this environment, so the notebook shows one default interactive state."))
        render_fn(**fallback)


print("Setup complete. Interactive controls enabled." if HAS_WIDGETS else "Setup complete. Static fallback mode.")


## 1. From fixed feature maps to learned representations

In a fixed-feature model, the design choice is the feature map and learning only selects coefficients:

$$
h_{\theta}(x)=\theta^{\top}\phi(x),
\qquad
\theta\in\mathbb{R}^{p}.
$$

The corresponding hypothesis space is

$$
\mathcal{H}_{\phi}
=\left\{h_{\theta}:h_{\theta}(x)=\theta^{\top}\phi(x),\ \theta\in\mathbb{R}^{p}\right\}.
$$

For a polynomial feature map,

$$
\phi_d(x)=\begin{bmatrix}1 & x & x^2 & \cdots & x^d\end{bmatrix}^{\top},
$$

changing $\theta$ changes the selected polynomial, while changing $d$ changes $\mathcal{H}$ itself.

A neural network learns intermediate representations. With $z_0=x$, a feed-forward network can be written as

$$
z_{\ell}=\sigma(W_{\ell}z_{\ell-1}+b_{\ell}),
\qquad \ell=1,\ldots,L-1,
$$

and

$$
h_{\theta}(x)=W_Lz_{L-1}+b_L.
$$

| Action | Fixed-feature model | Neural network |
|---|---|---|
| Change coefficients | Move within $\mathcal{H}_\phi$ | Move within $\mathcal{H}_{\mathrm{NN}}$ |
| Change degree or basis | Redefine $\mathcal{H}$ | Architecture-level change |
| Change hidden weights | Not available | Learn representation |
| Add units or layers | Redefine capacity and bias | Redefine $\mathcal{H}$ |

> **Researcher note.** Choosing $\phi$, width, depth, activation, or architecture is not a neutral implementation detail. It defines which explanations the learner can represent and which unsupported behaviours it will tend to fill in.


In [ ]:
show_interactive(
    "First change the coefficient preset: this selects a different function inside a fixed feature family. Then change degree or move the ReLU kinks: those actions change what $\mathcal{H}$ makes available or natural.",
    plot_fixed_vs_learned,
    controls={
        "degree": slider("IntSlider", value=3, min=1, max=9, step=1, description="polynomial degree"),
        "coefficient_preset": dropdown(options=["smooth", "oscillating", "tilted"], value="smooth", description="coefficients"),
        "kink_shift": slider("FloatSlider", value=0.0, min=-20.0, max=20.0, step=2.0, description="ReLU kink shift"),
        "output_scale": slider("FloatSlider", value=1.0, min=0.3, max=1.8, step=0.1, description="output weight scale"),
    },
    fallback={"degree": 3, "coefficient_preset": "smooth", "kink_shift": 0.0, "output_scale": 1.0},
)

markdown_table(["User action", "Ingredient changed"], fixed_vs_learned_action_rows())

display(Markdown(
    r"> **Checkpoint.** Did we change $\mathcal{D}$, $\mathcal{H}$, $\mathcal{O}$, the estimand, or the deployment distribution?"
))


## 2. One ReLU neuron as a learned nonlinear feature

A single neuron defines a learned feature

$$
a_{w,b}(x)=\sigma(w^{\top}x+b),
$$

where $w$ controls orientation and scale, $b$ shifts the activation boundary, and $\sigma$ is the activation function.

For ReLU,

$$
\sigma(t)=\max(0,t),
\qquad
a_{w,b}(x)=\max(0,wx+b).
$$

If $w\neq 0$, the kink occurs at

$$
x^{\star}=-\frac{b}{w}.
$$

Before running the cell, predict where the kink will move when $b$ increases while $w$ is fixed.

The nonlinearity is essential. Without $\sigma$, two linear layers collapse into one linear map:

$$
W_2(W_1x+b_1)+b_2=(W_2W_1)x+(W_2b_1+b_2).
$$

Stacking linear layers therefore does not create a richer hypothesis space.


In [ ]:
show_interactive(
    "Hold $w>0$ fixed and increase $b$. The kink $x^*=-b/w$ should move left. Then change the sign of $w$ and check how the active side flips.",
    plot_relu_neuron,
    controls={
        "w": slider("FloatSlider", value=0.08, min=-0.12, max=0.12, step=0.01, description="w"),
        "b": slider("FloatSlider", value=-3.6, min=-8.0, max=8.0, step=0.2, description="b"),
    },
    fallback={"w": 0.08, "b": -3.6},
)

display(Markdown(
    r"> **Researcher note.** A learned hidden weight is not just a coefficient on a fixed covariate. It changes where a nonlinear feature turns on, so it changes the shapes the learner can make natural."
))


## 3. Width: combining many learned features

A one-hidden-layer network with $m$ hidden units can be written as

$$
h(x)=c+\sum_{j=1}^{m}v_j\sigma(w_jx+b_j).
$$

For ReLU in one dimension,

$$
h(x)=c+\sum_{j=1}^{m}v_j\max(0,w_jx+b_j),
$$

which is a learned piecewise-linear function. Each active hidden unit contributes a hinge at

$$
x_j^{\star}=-\frac{b_j}{w_j},
$$

when $w_j\neq 0$. The width-limited hypothesis space is

$$
\mathcal{H}_{m}
=\left\{x\mapsto c+\sum_{j=1}^{m}v_j\sigma(w_jx+b_j):c,v_j,w_j,b_j\in\mathbb{R}\right\}.
$$

Increasing width changes the functions available and can reduce approximation error. But with finite data it can also enlarge the set of functions that fit the observations while disagreeing away from support.

Universal approximation results say that sufficiently wide networks can approximate broad classes of continuous functions on compact domains:

$$
\forall f\in C(K),\ \forall \varepsilon>0,\ \exists h\in\mathcal{H}_{m}
\text{ such that }
\sup_{x\in K}|f(x)-h(x)|<\varepsilon.
$$

This is an approximation statement, not an optimisation or generalisation guarantee.


In [ ]:
show_interactive(
    "Increase width to add available hinges. Move the focused unit to see which local contribution changes the summed function. Shift the kinks and ask whether the data would justify that movement in an unsupported region.",
    plot_width_demo,
    controls={
        "width": slider("IntSlider", value=4, min=1, max=10, step=1, description="width m"),
        "focus_unit": slider("IntSlider", value=1, min=1, max=10, step=1, description="focused unit"),
        "kink_shift": slider("FloatSlider", value=0.0, min=-18.0, max=18.0, step=2.0, description="kink shift"),
        "output_scale": slider("FloatSlider", value=1.0, min=0.2, max=1.8, step=0.1, description="output scale"),
    },
    fallback={"width": 4, "focus_unit": 1, "kink_shift": 0.0, "output_scale": 1.0},
)

display(Markdown(
    r"> **Checkpoint.** Did width change the data, the optimiser, or the hypothesis space? What behaviour would still be unsupported if the data had a gap?"
))


## 4. Depth: composition and reuse

A deep feed-forward network repeatedly transforms a representation:

$$
z_0=x,
\qquad
z_{\ell}=\sigma(W_{\ell}z_{\ell-1}+b_{\ell}),
\qquad \ell=1,\ldots,L-1,
$$

with output

$$
h_{\theta}(x)=W_Lz_{L-1}+b_L.
$$

Written as a composition,

$$
h_{\theta}(x)=g_L(g_{L-1}(\cdots g_1(x)\cdots)).
$$

Depth does not merely add parameters. It changes the factorisation of the function. Later layers reuse and recombine earlier features, making some compositional functions easier to represent than they would be in a shallow model.

For ReLU networks, each activation pattern defines a region $R_r$ of input space. Inside one such region, the network is affine:

$$
h_{\theta}(x)=A_rx+c_r,
\qquad x\in R_r.
$$


In [ ]:
show_interactive(
    "Change the first-layer scale to move early features, then change the recombination shift. Look for bends that appear because later layers reuse earlier features rather than starting again from $x$.",
    plot_depth_demo,
    controls={
        "first_layer_scale": slider("FloatSlider", value=1.0, min=0.4, max=1.8, step=0.1, description="first-layer scale"),
        "recombination_shift": slider("FloatSlider", value=0.0, min=-1.0, max=1.0, step=0.1, description="layer-2 bias shift"),
        "output_mix": slider("FloatSlider", value=1.0, min=0.3, max=1.7, step=0.1, description="output mix"),
    },
    fallback={"first_layer_scale": 1.0, "recombination_shift": 0.0, "output_mix": 1.0},
)

display(Markdown(
    r"> **Researcher note.** A deep model does not only have more adjustable numbers. It encodes a preference for functions that can be built by reusing intermediate representations."
))


## 5. Architecture as inductive bias

An architecture is not only an implementation choice. It changes the functions that are available, efficient, and natural inside $\mathcal{H}$.

| Architecture | Structural assumption | What becomes natural in $\mathcal{H}$ | Example diagnostic |
|---|---|---|---|
| MLP | Dense mixing | Generic nonlinear maps | Which coordinates interact? |
| CNN | Locality + translation equivariance | Shifted patterns share features | Shift input, output shifts. |
| RNN | Reused transition over time | Sequential state update | Same rule at each time step. |
| Transformer | Token-token interaction | Content-dependent mixing | Attention matrix. |
| GNN | Neighbourhood message passing | Relational/local graph computation | Node update from neighbours. |

The point is not that one architecture is universally better. The point is that each architecture makes some functions easier to represent and some unsupported behaviours more likely.


In [ ]:
show_interactive(
    "Choose one architecture diagnostic at a time. For CNNs, vary the shift; for attention, vary the query token; for GNNs, vary the message weight. Identify the structural assumption being encoded.",
    plot_architecture_demo,
    controls={
        "architecture": dropdown(options=["CNN local filter", "Transformer attention", "GNN message passing"], value="CNN local filter", description="architecture"),
        "shift": slider("IntSlider", value=6, min=0, max=15, step=1, description="CNN shift"),
        "query_token": slider("IntSlider", value=0, min=0, max=4, step=1, description="query token"),
        "message_weight": slider("FloatSlider", value=0.45, min=0.0, max=1.0, step=0.05, description="message weight"),
    },
    fallback={"architecture": "CNN local filter", "shift": 6, "query_token": 0, "message_weight": 0.45},
)

display(Markdown(
    r"> **Checkpoint.** Which assumption was encoded by the architecture: locality, translation, sequential reuse, token interaction, or graph neighbourhood structure?"
))


## 6. Data support, interpolation, and extrapolation

Let the observed dataset be

$$
\mathcal{D}=\{(x_i,y_i)\}_{i=1}^{n}.
$$

Finite data constrain a hypothesis only where observations provide evidence. The empirical risk is

$$
\widehat{R}_{\mathcal{D}}(h)
=\frac{1}{n}\sum_{i=1}^{n}\ell(h(x_i),y_i),
$$

so two functions can have similar empirical risk while disagreeing away from the observed inputs. A compatible set can be written as

$$
\{h\in\mathcal{H}:\widehat R_{\mathcal D}(h)\leq \epsilon\}.
$$

Inside support, the data constrain behaviour. Outside support, the model continues the function according to the assumptions encoded in $\mathcal{H}$ and the selection pressures in $\mathcal{O}$.

We now hold $\mathcal{D}$ fixed and vary $\mathcal{H}$.


In [ ]:
show_interactive(
    "Keep $\mathcal{D}$ fixed. Change only the hypothesis class and complexity, then compare the gap error with the observed-region error. The unsupported-region behaviour is being supplied by $\mathcal{H}$ and the ridge selection pressure.",
    plot_support_model,
    controls={
        "model_kind": dropdown(options=["polynomial", "ReLU basis", "periodic basis"], value="polynomial", description="hypothesis class"),
        "complexity": slider("IntSlider", value=5, min=1, max=18, step=1, description="complexity"),
        "lam": slider("FloatLogSlider", value=0.02, base=10, min=-5, max=0, step=0.25, description="ridge lambda"),
    },
    fallback={"model_kind": "polynomial", "complexity": 5, "lam": 0.02},
)

display(Markdown("Oracle errors use the simulated reference curve and would not be available in ordinary research settings."))
markdown_table(["Ingredient", "Held fixed?", "Changed?", "Consequence"], what_changed_rows())

display(Markdown(
    r"> **Researcher note.** Agreement near observations is not evidence that a model has recovered the intended response in the gap. The gap behaviour is supplied by $\mathcal{H}$ and $\mathcal{O}$, not by observed samples."
))


## 7. What expressivity does not solve

Expressivity asks whether the target function can be represented inside the hypothesis space:

$$
\inf_{h\in\mathcal{H}}R(h),
\qquad
R(h)=\mathbb{E}_{(X,Y)\sim P}\left[\ell(h(X),Y)\right].
$$

But this is only one issue. The learning problem also involves:

- representation: is there an $h\in\mathcal{H}$ with small risk?
- optimisation: can training find a good parameter vector?

$$
\widehat{\theta}\approx\arg\min_{\theta\in\Theta}\widehat{R}_{\mathcal{D}}(h_{\theta}).
$$

- identification: can finite data distinguish between plausible functions?

$$
h_1(x_i)=h_2(x_i)\ \forall i
\quad\not\Rightarrow\quad
h_1(x)=h_2(x)\ \forall x.
$$

- generalisation: will empirical performance transfer to new data?

$$
R(\widehat{h})-\widehat{R}_{\mathcal{D}}(\widehat{h}).
$$

Exit question: a model class can represent the true function. Why might the learned solution still be scientifically unreliable?

Expected answers include incomplete support, finite-data non-identification, optimiser choice, validation that misses deployment conditions, and an estimand that does not match the research claim.


In [ ]:
show_interactive(
    "Move the selected degree and watch the capacity marker. Then turn on random labels: if the model can fit them, low training error alone cannot be the research claim.",
    plot_capacity_demo,
    controls={
        "degree": slider("IntSlider", value=5, min=0, max=17, step=1, description="selected degree"),
        "show_random_labels": checkbox(value=False, description="show random-label condition"),
    },
    fallback={"degree": 5, "show_random_labels": False},
)

display(Markdown(
    r"> **Checkpoint.** Did better training fit change $\mathcal{D}$, $\mathcal{H}$, $\mathcal{O}$, the estimand, or the deployment distribution? Which validity claim is actually being tested?"
))


## 8. Parameter space versus hypothesis space

The parameter vector $\theta$ is a coordinate in parameter space $\Theta$. The realised function $h_{\theta}$ is an element of hypothesis space $\mathcal{H}$. The map

$$
q:\Theta\to\mathcal{H},
\qquad
q(\theta)=h_{\theta},
$$

need not be one-to-one. Different parameter values can represent the same function.

For a one-hidden-layer network,

$$
h(x)=c+\sum_{j=1}^{m}v_j\sigma(w_jx+b_j),
$$

permuting hidden units leaves the function unchanged. ReLU also has positive homogeneity. For $\alpha>0$,

$$
v_j\sigma(w_jx+b_j)
=\frac{v_j}{\alpha}\sigma(\alpha w_jx+\alpha b_j).
$$

Optimisation moves through $\Theta$, but prediction and research claims are usually about function behaviour.


In [ ]:
show_interactive(
    "Switch between permutation and positive rescaling. The plotted function should not move, even though the parameter vector changed.",
    plot_parameter_equivalence,
    controls={
        "transform": dropdown(options=["permutation", "positive rescaling"], value="permutation", description="parameter transform"),
        "alpha": slider("FloatSlider", value=2.0, min=0.25, max=4.0, step=0.25, description="rescale alpha"),
    },
    fallback={"transform": "permutation", "alpha": 2.0},
)

display(Markdown(
    r"> **Researcher note.** A parameter-space story is not automatically a function-space story. Different optimisation paths can be different coordinates for the same predictive behaviour."
))


## 9. Summary and handoff to optimisation

The workshop frame remains

$$
\boxed{\mathcal{H}}+\mathcal{D}+\mathcal{O}\rightarrow s.
$$

In this notebook, $\mathcal{H}$ defined what could be represented and what behaviour was natural when evidence was incomplete. The data $\mathcal{D}$ constrained behaviour where observations existed. The optimisation procedure $\mathcal{O}$ would still be needed to select one member of the compatible set.

A typical selection rule has the form

$$
\widehat{h}
=\arg\min_{h\in\mathcal{H}}
\left[\widehat{R}_{\mathcal{D}}(h)+\lambda\Omega(h)\right].
$$

The optimisation notebook asks which member of $\mathcal{H}$ is actually selected once we choose a loss, optimiser, regularisation, initialisation, and stopping rule.

Failure diagnosis exercise:

| Observation | Likely source | What to check next |
|---|---|---|
| Good random-test error, bad shifted-context error | Data / robustness | Compare $P_{\mathrm{train}}$ and $P_{\mathrm{deploy}}$. |
| Models disagree in a data gap | Data support + hypothesis bias | Inspect coverage and version-space disagreement. |
| Training loss high for all capacities | Hypothesis or optimisation | Separate approximation failure from training failure. |
| Training loss low, test loss high | Generalisation or distribution mismatch | Inspect validation design, regularisation, and support. |
| Same data and $\mathcal{H}$, different solutions | Optimisation | Inspect initialisation, optimiser, stochasticity, and stopping. |


In [ ]:
markdown_table(["Ingredient", "Notebook examples", "Research question"], summary_rows())

display(Markdown(
    r"> **Checkpoint.** What changed: $\mathcal{D}$, $\mathcal{H}$, $\mathcal{O}$, the estimand, or the deployment distribution? Use that question before diagnosing any model result."
))
